# team_goals_against — one model, two terms (clean_sheet + conceded)

**Render-not-decide.** A re-runnable view of the joint defensive model, **not** the record — the frozen numbers live in [predictive-phase3-points-model.md](../../../docs/studies/results/predictive-phase3-points-model.md). All logic is in `model/terms/team_goals_against/`; the notebook only calls it and plots.

D-D: `clean_sheet = 1{GA=0}` and `conceded = -floor(GA/2)` fall out of **one** Poisson mean per team-fixture (`lambda_ga`). Population: `minutes > 0`, DGW excluded, GW > 3, conditional on appearance (see `ASSUMPTIONS.md`).

## Setup

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart
from model.terms.team_goals_against import CleanSheetTerm, ConcededTerm, TeamGoalsAgainstModel

mart = load_mart()
model = TeamGoalsAgainstModel(variant="selected")  # ONE model shared by both terms
cs_term, conc_term = CleanSheetTerm(model), ConcededTerm(model)
[f.name for f in model.pool.candidates], model.pool.minimal

## Pre-fit assumptions
> Is Poisson justified on team GA (material dispersion ~1) and is the effect learnable at this N? See `ASSUMPTIONS.md` §2, §7.

In [ ]:
report = model.check_assumptions(TeamGoalsAgainstModel.population(mart))
print(f"family_ok={report.family_ok}  detectable={report.detectable}  n_train={report.n_train}")
report.dispersion

## The joint emit — both terms from one fit
> Fit once; emit returns `clean_sheet` (p_cs) and `conceded` (e_conceded_pts), internally consistent by construction (P(CS)=1 cannot coexist with an expected penalty).

In [ ]:
fitted = model.fit(mart)
emitted = model.emit(fitted)
list(emitted)  # -> ['clean_sheet', 'conceded']

## Gates — each term vs its own naive bar (spec §5)
> clean_sheet vs lagged `clean_sheets_roll3` (GK/DEF/MID); conceded vs the lagged-GA-implied penalty (GK/DEF). Broadcast team_gw -> player_gw first.

In [ ]:
cs_gate, conc_gate = cs_term.validate(mart), conc_term.validate(mart)
display(cs_gate.table)
display(conc_gate.table)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
cs_gate.table.set_index("position")[["baseline", "p_cs"]].plot.bar(ax=axes[0], title="clean_sheet: model vs baseline")
conc_gate.table.set_index("position")[["baseline", "e_conceded"]].plot.bar(ax=axes[1], title="conceded: model vs baseline")
for ax in axes:
    ax.set_ylabel("within-position Spearman")
plt.tight_layout()

## Diagnose — residuals + ablation (post-gate)
> Which fixtures the model misses, and the measured contribution of each feature (drop-one, re-gate).

In [ ]:
diag = cs_term.diagnose(mart)
display(diag.ablation)
display(diag.residuals.head(10))